In [23]:
GITHUB_URL = "https://models.github.ai/inference"
OPENROUTER_URL = "https://openrouter.ai/api/v1"

In [30]:
import os
from dotenv import load_dotenv

# โหลดค่าจากไฟล์ .env
load_dotenv()

# ดึงค่ามาใช้
GROQ_API_KEY = os.getenv("GROQ_KEY")
OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")
GITHUB_KEY = os.getenv("GITHUB_KEY")

## FILE PATHS

In [2]:
SEC_JSON_PATH = 'intemediate_data/sec_reg_structured.json'
SEC_W_KW_PATH = 'intemediate_data/sec_reg_tree_enhanced.json'
# QUESTION_JSON_PATH = 'questions/eval_questions2.json'
# TREERAG_RESULT_PATH = 'results/legaltreerag_ans.csv'

## LLM MODELS

In [19]:
# KEYWORD_EXTRACTION_MODEL = "llama-3.1-8b-instant"
# KEYWORD_EXTRACTION_MODEL = "openai/gpt-oss-120b"
KEYWORD_EXTRACTION_MODEL = "llama-3.3-70b-versatile"

# KEYWORD_EXTRACTION_MODEL = "openai/gpt-oss-120b:free"

In [15]:


# llm_model_retrieve = "openai/gpt-4.1"
# llm_model_ans = "openai/gpt-4.1-mini"
# llm_model_ans = "openai/gpt-4.1"

# llm_model_retrieve = "deepseek/deepseek-r1"
# llm_model_ans = "deepseek/deepseek-v3"

# llm_model_retrieve = "Llama-3.3-70B-Instruct"
# llm_model_ans = "meta/llama-3.3-70b-instruct"



# llm_model_retrieve = "llama-3.3-70b-versatile"
# llm_model_retrieve = "openai/gpt-oss-safeguard-20b"
# llm_model_retrieve = "openai/gpt-oss-120b"
# llm_model_ans = "meta-llama/llama-4-scout-17b-16e-instruct"

# llm_model_retrieve = "llama-3.1-8b-instant"
# llm_model_ans = 'openai/gpt-oss-120b'
# llm_model_ans = "qwen/qwen3-32b"



In [ ]:
# from openai import OpenAI
# import json
# import re

# client = OpenAI(base_url=OPENROUTER__MODEL_URL, api_key=OPENROUTER_KEY,)
# client = OpenAI(base_url=GITHUB_MODEL_URL, api_key=GITHUB_KEY)


In [20]:

from groq import Groq

os.environ["GROQ_API_KEY"] = GROQ_KEY
client = Groq()

## Phase 1.1 : Using LLM to add Keywords to each nodes

In [22]:
import json
import time
import re


# โหลดข้อมูล
try:
    with open(SEC_JSON_PATH, 'r', encoding='utf-8') as f:
        reg_data = json.load(f)
except FileNotFoundError:
    print(f"❌ ไม่พบไฟล์ {SEC_JSON_PATH} กรุณาตรวจสอบชื่อไฟล์อีกครั้ง")
    exit()

# ---------------------------------------------------------
# 1. ฟังก์ชันช่วย: กำหนด ID แบบ Hierarchical
# ---------------------------------------------------------
def assign_node_ids(nodes, parent_id="root"):
    for i, node in enumerate(nodes):
        current_id = f"{parent_id}_{i}"
        node['node_id'] = current_id
        if 'subsection' in node and len(node['subsection']) > 0:
            assign_node_ids(node['subsection'], current_id)

# ---------------------------------------------------------
# 2. ค้นหาและรวบรวม Node ที่ต้องทำ Keyword ลง List
# ---------------------------------------------------------
target_nodes = []
target_levels = ['ภาค', 'หมวด', 'ข้อ', 'ส่วนนำ', 'ส่วน', "คำศัพท์"]

def collect_target_nodes(nodes):
    for node in nodes:
        if node.get('section_type') in target_levels:
            target_nodes.append(node)
        if 'subsection' in node and len(node['subsection']) > 0:
            collect_target_nodes(node['subsection'])

# ---------------------------------------------------------
# 3. ฟังก์ชันหลัก: เรียก LLM แบบมัดรวม (Batch Processing) พร้อมระบบ Retry
# ---------------------------------------------------------
def process_keywords_in_batches(node_list, batch_size=10): 
    total_nodes = len(node_list)
    print(f"📦 พบหัวข้อที่ต้องสกัด Keyword ทั้งหมด {total_nodes} ข้อ (แบ่งทำทีละ {batch_size} ข้อ)")
    
    for i in range(0, total_nodes, batch_size):
        batch = node_list[i:i+batch_size]
        print(f"\n⏳ กำลังสกัด Keyword ชุดที่ {i//batch_size + 1} (ข้อ {i+1} ถึง {min(i+batch_size, total_nodes)})...")
        
        # 1. ดึง ID ทั้งหมดใน Batch นี้เพื่อเอาไปบังคับ LLM
        batch_ids = [node['node_id'] for node in batch]
        
        prompt_text = "จงสกัด Keyword สำคัญทางกฎหมาย 8-15 คำ ของแต่ละหัวข้อต่อไปนี้ โดยต้องครอบคลุมถึงรายละเอียดในข้อย่อยด้วย:\n\n"
        for node in batch:
            node_id = node['node_id']
            section = node.get('section', '')
            raw_content = node.get('raw', node.get('description', ''))
            content_to_use = " ".join(raw_content.split())[:2000] 
            
            prompt_text += f"ID: {node_id}\nหัวข้อ: {section}\nเนื้อหา: {content_to_use}\n---\n"
            
        prompt_text += f"""
คำสั่งและข้อบังคับ: 
1. ให้ตอบกลับเป็น JSON Object เท่านั้น โดยใช้ ID เป็น Key และ Keyword เป็น Value
2. Keyword ต้องประกอบด้วย "คำนามเฉพาะ", "ชื่อกลุ่มเป้าหมาย (เช่น ผู้ลงทุนสถาบัน, PP10)", และ "เงื่อนไขสำคัญ (เช่น ปรับโครงสร้างหนี้)" ที่ซ่อนอยู่ในข้อย่อยต่างๆ
3. ห้ามใช้คำกริยากว้างๆ คำเดียว และต้องสกัดให้ครอบคลุมเนื้อหาสำคัญทั้งหมดที่ปรากฏในข้อนั้น
4. **สำคัญมาก: คุณต้องสร้าง Key สำหรับ ID เหล่านี้ให้ครบทุกตัว ห้ามข้ามเด็ดขาด:**
{batch_ids}

ตัวอย่างรูปแบบการตอบ (ต้องมีครบทุก ID ที่ให้ไป):
{{
    "{batch_ids[0] if len(batch_ids) > 0 else 'root_1_0'}": "keyword1, keyword2, keyword3...",
    "{batch_ids[1] if len(batch_ids) > 1 else 'root_1_1'}": "keyword1, keyword2, keyword3..."
}}
"""
        
        # 📌 ระบบป้องกันความผิดพลาด (Retry Logic)
        MAX_RETRIES = 3 # ให้โอกาสโมเดลแก้ตัว 3 รอบ
        for attempt in range(MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt_text}],
                    # model="llama-3.1-8b-instant", # แนะนำให้เปลี่ยนเป็นโมเดลที่เสถียรที่สุดที่คุณมีโควต้า
                    model=KEYWORD_EXTRACTION_MODEL,
                    temperature=0.0, # บังคับ 0 เพื่อไม่ให้โมเดลคิดอะไรนอกกรอบ
                    response_format={"type": "json_object"}
                )
                
                result_json = json.loads(response.choices[0].message.content.strip())
                
                # เช็คว่ามี ID ไหนที่ส่งไปแล้วไม่ได้ตอบกลับมาบ้าง
                missing_keys = [node['node_id'] for node in batch if node['node_id'] not in result_json]
                
                if missing_keys:
                    print(f"  ⚠️ [Warning] Attempt {attempt + 1}: โมเดลลืมตอบ Keys เหล่านี้: {missing_keys}")
                    if attempt < MAX_RETRIES - 1:
                        print("  🔄 กำลังส่งคำสั่งเพื่อลองใหม่...")
                        time.sleep(2) # พัก API เล็กน้อยก่อนยิงซ้ำ
                        continue # วนลูปกลับไปยิง API ใหม่
                    else:
                        print("  ❌ พยายามครบโควต้าแล้ว ข้าม Keys ที่หายไป")
                
                # หากดึงข้อมูลสำเร็จ หรือพยายามครบทุกรอบแล้ว ให้นำผลลัพธ์ไปใส่ใน Node
                for node in batch:
                    node_id = node['node_id']
                    # ใช้ .get() เพื่อป้องกัน Error หากสุดท้าย Key ก็ยังหายอยู่ดี
                    node['keywords'] = result_json.get(node_id, "")
                    print(node_id, node['keywords'])
                
                print(f"  ✅ สำเร็จ!")
                break # ได้ผลลัพธ์ครบ จบการทำงานของ Attempt Loop นี้ เพื่อไปทำ Batch ถัดไป
                
            except Exception as e:
                print(f"  ⚠️ [Error] ชุดที่ {i//batch_size + 1} ล้มเหลว (Attempt {attempt+1}): {e}")
                if attempt == MAX_RETRIES - 1:
                    print("  ❌ พยายามครบโควต้าแล้ว บังคับให้ Keyword เป็นค่าว่าง")
                    for node in batch:
                        node['keywords'] = "" 
                time.sleep(3) # พักก่อนลองใหม่ กรณี API ล่ม

        # time.sleep(1) # ปลดคอมเมนต์บรรทัดนี้ได้หากติด Rate Limit ของ Groq

# ---------------------------------------------------------
# 4. เริ่มรันการทำงาน
# ---------------------------------------------------------
print("🚀 สเต็ป 1: กำหนด Node ID ให้ทั้งหมด...")
assign_node_ids(reg_data['data'])

print("🚀 สเต็ป 2: รวบรวมหัวข้อที่ต้องการ...")
collect_target_nodes(reg_data['data'])

print("🚀 สเต็ป 3: ส่ง LLM สกัด Keyword แบบ Batch...")
# แนะนำ Batch Size ประมาณ 20-30 กำลังดี ไม่ให้ Token ขาออกล้นเกินไป
# process_keywords_in_batches(target_nodes, batch_size=12) 
process_keywords_in_batches(target_nodes, batch_size=10)

# บันทึกไฟล์
with open(SEC_W_KW_PATH, 'w', encoding='utf-8') as f:
    json.dump(reg_data, f, ensure_ascii=False, indent=4)

print(f"\n🎉 สำเร็จ! บันทึกไฟล์ Tree แบบ Fast-Batch (จาก Raw Text) ลงใน '{output_filename}' เรียบร้อยแล้ว!\n")

🚀 สเต็ป 1: กำหนด Node ID ให้ทั้งหมด...
🚀 สเต็ป 2: รวบรวมหัวข้อที่ต้องการ...
🚀 สเต็ป 3: ส่ง LLM สกัด Keyword แบบ Batch...
📦 พบหัวข้อที่ต้องสกัด Keyword ทั้งหมด 222 ข้อ (แบ่งทำทีละ 10 ข้อ)

⏳ กำลังสกัด Keyword ชุดที่ 1 (ข้อ 1 ถึง 10)...
root_0_0 พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์, ผู้ออกหลักทรัพย์, ผู้ลงทุน, การออกและเสนอขายตราสารหนี้
root_0_1 วันที่ใช้บังคับ, วันที่ 1 ตุลาคม พ.ศ. 2565, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2 การกำหนดบทนิยาม, ตราสารหนี้, กองทรัสต์, หุ้นกู้แปลงสภาพ, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2_0 การกำหนดบทนิยาม, ตราสารหนี้, กองทรัสต์, หุ้นกู้แปลงสภาพ, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2_1 ตราสารหนี้, หุ้นกู้, พันธบัตร, ตั๋วเงิน, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2_1_0 ตราสารหนี้, หุ้นกู้, พันธบัตร, ตั๋วเงิน, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2_2 กองทรัสต์, ทรัสต์เพื่อการลงทุนในอสังหาริมทรัพย์, ทรัสต์เพื่อการลงทุนในโครงสร้างพื้นฐาน, ผู้ออกหลักทรัพย์, ผู้ลงทุน
root_0_2_2_0 กองทรัสต์, ทรัสต์เพื่อการลงทุนในอสังหาริมทรัพย์, ทรัสต์เพื่อการลงทุนในโครงสร้างพื้นฐาน, ผู้อ

NameError: name 'output_filename' is not defined